# PARTIE III — RNN, LSTM, GRU et Seq2Seq
**Module : Deep Learning — EMSI Casablanca 2025–2026**

**Tâche :** Modélisation de séquences textuelles — classification du sentiment  
**Dataset :** Avis Blend Gourmet Burger Casablanca (365 avis, FR/EN/AR)  
**Modèles :** RNN simple → LSTM → GRU → Seq2Seq (encodeur-décodeur)  
**Métriques :** Perplexité, Accuracy, F1-score, BLEU (Seq2Seq)


## 1. Théorie — Modèles de Langage et Architectures Récurrentes

### 1.1 Modèle de Langage et Factorisation par la Règle de Chaîne
Un modèle de langage estime la probabilité d'une séquence :

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})$$

À chaque pas, le modèle prédit le token suivant **conditionné** sur tout le contexte précédent.

### 1.2 Perplexité
La **perplexité** mesure l'incertitude moyenne du modèle :

$$\text{PPL} = \exp\left(-\frac{1}{T} \sum_{t=1}^{T} \log P(x_t \mid \text{contexte})\right)$$

- PPL = 1 : modèle parfait (certitude absolue)
- PPL = |V| : modèle aléatoire (vocabulaire de taille V)
- **Interprétation** : PPL = 10 signifie qu'à chaque position, le modèle hésite entre ≈10 tokens possibles.

### 1.3 RNN Simple
$$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$$
$$\hat{y}_t = W_{hy} h_t + b_y$$

**Problème :** le gradient vanishing — $\partial h_T / \partial h_0$ contient T multiplications de matrices → tend vers 0 ou ∞.

### 1.4 LSTM — Long Short-Term Memory
```
Porte d'oubli   : f_t = σ(W_f [h_{t-1}, x_t] + b_f)
Porte d'entrée  : i_t = σ(W_i [h_{t-1}, x_t] + b_i)
Candidat cellule: g_t = tanh(W_g [h_{t-1}, x_t] + b_g)
Cellule (LT)    : c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
Porte de sortie : o_t = σ(W_o [h_{t-1}, x_t] + b_o)
État caché      : h_t = o_t ⊙ tanh(c_t)
```
Le chemin du gradient passe par **c_t** via des additions → gradient stable.

### 1.5 GRU — Gated Recurrent Unit
```
Porte de reset  : r_t = σ(W_r [h_{t-1}, x_t])
Porte de màj    : z_t = σ(W_z [h_{t-1}, x_t])
Candidat        : ñ_t = tanh(W_n [r_t ⊙ h_{t-1}, x_t])
État            : h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ ñ_t
```
GRU fusionne la cellule et l'état caché → **moins de paramètres** que LSTM, souvent performances comparables.

### 1.6 BPTT et Gradient Clipping
La **Backpropagation Through Time (BPTT)** déplie le RNN sur T étapes.  
Le **gradient clipping** évite l'explosion : si $\|g\|_2 > \text{seuil}$ → $g \leftarrow g \cdot \text{seuil}/\|g\|_2$

### 1.7 Seq2Seq (Encodeur-Décodeur)
- **Encodeur** : lit la séquence source token par token → produit un vecteur de contexte $c$ (dernier état caché)
- **Décodeur** : génère la séquence cible conditionné par $c$
- **Teacher Forcing** : pendant l'entraînement, on donne le token cible réel (pas la prédiction) en entrée du décodeur → entraînement plus rapide et stable


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import os, re, math, time
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.nn.utils import clip_grad_norm_

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report)

os.makedirs("outputs", exist_ok=True)

# CONFIG
DATA_PATH  = "../data/reviews/blend_reviews_full.csv"
SEED       = 42
EPOCHS     = 40
LR         = 1e-3
BATCH_SIZE = 32
MAX_LEN    = 30
VOCAB_SIZE = 2000
EMBED_DIM  = 64
HIDDEN_DIM = 128
CLIP_GRAD  = 1.0

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
print("=" * 60)


## 2. Préparation des données — Tokenisation, Vocabulaire, Padding

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 2. PRÉPARATION DES DONNÉES TEXTUELLES
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f"Dataset : {len(df)} avis  |  Colonnes : {list(df.columns)}")
print(f"Distribution des sentiments :\n{df['sentiment'].value_counts()}")

# ── Tokens spéciaux ─────────────────────────────────────────────────────────
PAD_IDX = 0   # padding
UNK_IDX = 1   # token inconnu
SOS_IDX = 2   # start of sequence (Seq2Seq)
EOS_IDX = 3   # end of sequence   (Seq2Seq)

STOP_WORDS = {
    "le","la","les","de","du","des","un","une","et","ou","à","au","aux","en",
    "dans","sur","par","pour","avec","sans","très","plus","bien","tout","ce",
    "the","a","an","and","or","in","on","at","to","for","of","is","it","we","i"
}

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return [t for t in text.split() if t not in STOP_WORDS and len(t) > 1]

# Vocabulaire (fit sur tout le corpus)
all_tokens = []
for text in df["texte"].fillna(""):
    all_tokens.extend(tokenize(text))
vocab_counter = Counter(all_tokens)
vocab    = ["<PAD>","<UNK>","<SOS>","<EOS>"] +            [w for w, _ in vocab_counter.most_common(VOCAB_SIZE - 4)]
word2idx = {w: i for i, w in enumerate(vocab)}
VOCAB_REAL = len(vocab)
print(f"\nVocabulaire réel : {VOCAB_REAL} tokens")
print(f"Tokens spéciaux  : <PAD>={PAD_IDX}  <UNK>={UNK_IDX}  <SOS>={SOS_IDX}  <EOS>={EOS_IDX}")

def encode_seq(text, max_len=MAX_LEN):
    """Tokenise, encode, tronque et padde une séquence."""
    tokens = tokenize(text)[:max_len]
    ids    = [word2idx.get(t, UNK_IDX) for t in tokens]
    # Masquage implicite : padding à droite
    ids   += [PAD_IDX] * (max_len - len(ids))
    return ids

# ── Labels ──────────────────────────────────────────────────────────────────
le = LabelEncoder()
y  = le.fit_transform(df["sentiment"])
CLASSES  = list(le.classes_)
N_CLS    = len(CLASSES)
print(f"Classes : {CLASSES}")

# ── Encodage ────────────────────────────────────────────────────────────────
X = np.array([encode_seq(t) for t in df["texte"].fillna("")])
y = np.array(y)

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15,
                                               random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15/0.85,
                                                   random_state=SEED, stratify=y_tv)
print(f"\nTrain : {len(X_train)}  |  Val : {len(X_val)}  |  Test : {len(X_test)}")

def make_loader(X, y, shuffle=False):
    ds = TensorDataset(torch.tensor(X, dtype=torch.long),
                       torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

train_loader = make_loader(X_train, y_train, shuffle=True)
val_loader   = make_loader(X_val,   y_val)
test_loader  = make_loader(X_test,  y_test)


## 3. Implémentation RNN, LSTM, GRU — Comparaison

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 3. MODÈLES RÉCURRENTS
# ─────────────────────────────────────────────────────────────────────────────
class RecurrentClassifier(nn.Module):
    """
    Classifieur de sentiment basé sur RNN / LSTM / GRU.
    
    Architecture :
      Embedding → RNN/LSTM/GRU → dernier état caché → FC → N_CLS
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, n_classes,
                 rnn_type="lstm", n_layers=2, dropout=0.3):
        super().__init__()
        self.rnn_type  = rnn_type.lower()
        self.embed     = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        
        rnn_kwargs = dict(
            input_size=embed_dim, hidden_size=hidden_dim,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=False
        )
        if self.rnn_type == "rnn":
            self.rnn = nn.RNN(**rnn_kwargs)
        elif self.rnn_type == "lstm":
            self.rnn = nn.LSTM(**rnn_kwargs)
        elif self.rnn_type == "gru":
            self.rnn = nn.GRU(**rnn_kwargs)
        else:
            raise ValueError(f"rnn_type inconnu : {rnn_type}")
        
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden_dim, n_classes)

    def forward(self, x):
        emb = self.drop(self.embed(x))                # (B, L, E)
        if self.rnn_type == "lstm":
            out, (h, c) = self.rnn(emb)
        else:
            out, h = self.rnn(emb)
        # Dernier état caché de la dernière couche
        h_last = h[-1]                                 # (B, H)
        return self.fc(self.drop(h_last))

model_rnn  = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, rnn_type="rnn")
model_lstm = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, rnn_type="lstm")
model_gru  = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS, rnn_type="gru")

for name, m in [("RNN", model_rnn), ("LSTM", model_lstm), ("GRU", model_gru)]:
    n = sum(p.numel() for p in m.parameters())
    print(f"  {name:6s} : {n:,} paramètres")


## 4. BPTT et Gradient Clipping — Démonstration expérimentale

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4. BPTT ET GRADIENT CLIPPING — ILLUSTRATION
# ─────────────────────────────────────────────────────────────────────────────
print("Illustration de l'effet du gradient clipping sur le RNN...")

def measure_grad_norms(model, clip=None, n_batches=20):
    """Mesure les normes de gradient avec et sans clipping."""
    model = model.to(device)
    opt = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    grad_norms = []

    model.train()
    for i, (xb, yb) in enumerate(train_loader):
        if i >= n_batches: break
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = loss_fn(out, yb)
        loss.backward()

        # Norme du gradient AVANT clipping
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        grad_norms.append(total_norm)

        if clip is not None:
            clip_grad_norm_(model.parameters(), max_norm=clip)
        opt.step()

    return grad_norms

rnn_test_no_clip = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS,
                                        rnn_type="rnn").to(device)
rnn_test_clip    = RecurrentClassifier(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM, N_CLS,
                                        rnn_type="rnn").to(device)

norms_no_clip = measure_grad_norms(rnn_test_no_clip, clip=None,   n_batches=30)
norms_clip    = measure_grad_norms(rnn_test_clip,    clip=CLIP_GRAD, n_batches=30)

plt.figure(figsize=(10, 4))
plt.plot(norms_no_clip, label="Sans clipping", color="#E74C3C", alpha=0.8)
plt.plot(norms_clip,    label=f"Avec clipping (max={CLIP_GRAD})", color="#2ECC71", alpha=0.8)
plt.axhline(CLIP_GRAD, linestyle="--", color="#2ECC71", alpha=0.4)
plt.title("Norme L2 du gradient — Effet du Gradient Clipping", fontweight="bold")
plt.xlabel("Batch"); plt.ylabel("‖g‖₂"); plt.legend(); plt.grid(alpha=0.3)
plt.savefig("outputs/p3_gradient_clipping.png", dpi=120)
plt.show()
print("→ Figure sauvegardée : outputs/p3_gradient_clipping.png")
print(f"\nSans clipping — max norme : {max(norms_no_clip):.4f}")
print(f"Avec clipping — max norme : {max(norms_clip):.4f}  (≤ {CLIP_GRAD}) ✓")


## 5. Entraînement et Comparaison RNN vs LSTM vs GRU

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5. ENTRAÎNEMENT COMPARATIF
# ─────────────────────────────────────────────────────────────────────────────
def compute_perplexity(loss):
    """Perplexité = exp(cross-entropy loss)."""
    return math.exp(min(loss, 20))  # cap pour éviter overflow

def train_rnn_model(model, name, epochs=EPOCHS, clip=CLIP_GRAD):
    model = model.to(device)
    opt     = optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss":[], "val_loss":[], "val_acc":[], "train_ppl":[], "val_ppl":[]}
    best_acc, best_state = 0, None

    for ep in range(epochs):
        # ── Train
        model.train()
        tl, n_samples = 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            clip_grad_norm_(model.parameters(), max_norm=clip)
            opt.step()
            tl += loss.item() * len(yb)
            n_samples += len(yb)
        tl /= n_samples
        history["train_loss"].append(tl)
        history["train_ppl"].append(compute_perplexity(tl))

        # ── Val
        model.eval()
        vl, correct, total = 0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out  = model(xb)
                loss = loss_fn(out, yb)
                vl      += loss.item() * len(yb)
                correct += (out.argmax(1) == yb).sum().item()
                total   += len(yb)
        vl  /= total
        vacc = correct / total
        history["val_loss"].append(vl)
        history["val_acc"].append(vacc)
        history["val_ppl"].append(compute_perplexity(vl))

        if vacc > best_acc:
            best_acc   = vacc
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (ep + 1) % 10 == 0:
            print(f"  [{name}] Ep {ep+1:2d}/{epochs}"
                  f"  train_loss={tl:.4f}  val_loss={vl:.4f}"
                  f"  val_acc={vacc:.4f}  val_PPL={history['val_ppl'][-1]:.2f}")

    model.load_state_dict(best_state)
    torch.save(best_state, f"outputs/best_{name.lower()}_partie3.pt")
    print(f"  [{name}] → best_val_acc={best_acc:.4f}  sauvegardé\n")
    return history, best_acc

histories = {}
for name, model in [("RNN", model_rnn), ("LSTM", model_lstm), ("GRU", model_gru)]:
    print(f"Entraînement {name}...")
    hist, best = train_rnn_model(model, name)
    histories[name] = (hist, best)


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6. ÉVALUATION + VISUALISATION COMPARATIVE
# ─────────────────────────────────────────────────────────────────────────────
COLORS = {"RNN": "#E74C3C", "LSTM": "#3498DB", "GRU": "#2ECC71"}

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, wspace=0.35, hspace=0.4)

ax_tl  = fig.add_subplot(gs[0, 0])
ax_vl  = fig.add_subplot(gs[0, 1])
ax_ppl = fig.add_subplot(gs[0, 2])
ax_va  = fig.add_subplot(gs[1, 0])
ax_cm  = fig.add_subplot(gs[1, 1])
ax_bar = fig.add_subplot(gs[1, 2])

for name, (hist, _) in histories.items():
    c = COLORS[name]
    ax_tl.plot(hist["train_loss"],  label=name, color=c)
    ax_vl.plot(hist["val_loss"],    label=name, color=c)
    ax_ppl.plot(hist["val_ppl"],    label=name, color=c)
    ax_va.plot(hist["val_acc"],     label=name, color=c)

for ax, title in [(ax_tl,"Train Loss"), (ax_vl,"Val Loss"),
                   (ax_ppl,"Val Perplexité"), (ax_va,"Val Accuracy")]:
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3); ax.set_xlabel("Epoch")

# Matrice de confusion — LSTM (meilleur attendu)
model_lstm.eval()
preds_lstm, labels_all = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        out = model_lstm(xb.to(device))
        preds_lstm.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(yb.numpy())
cm = confusion_matrix(labels_all, preds_lstm)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax_cm, cbar=False)
ax_cm.set_title("Matrice de Confusion — LSTM")
ax_cm.set_xlabel("Prédit"); ax_cm.set_ylabel("Réel")

# Bar chart comparatif
names   = list(histories.keys())
accs    = [histories[n][1] for n in names]
n_params = [sum(p.numel() for p in m.parameters())
            for m in [model_rnn, model_lstm, model_gru]]
ax_bar.bar(names, accs, color=[COLORS[n] for n in names], alpha=0.85, edgecolor="white")
for i, (a, np_) in enumerate(zip(accs, n_params)):
    ax_bar.text(i, a + 0.005, f"{a:.3f}\n({np_//1000}K)", ha="center", fontsize=9)
ax_bar.set_ylim(0, 1); ax_bar.set_title("Accuracy & Paramètres"); ax_bar.grid(axis="y", alpha=0.3)

fig.suptitle("Comparaison RNN / LSTM / GRU — Sentiment Avis Blend Burger",
             fontsize=13, fontweight="bold")
plt.savefig("outputs/p3_rnn_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nTableau comparatif final :")
print(f"  {'Modèle':8s}  {'Val Acc':>8}  {'Val PPL':>8}  {'Paramètres':>12}")
print("  " + "-" * 42)
for name, (hist, best) in histories.items():
    final_ppl = hist["val_ppl"][-1]
    np_  = sum(p.numel() for p in {"RNN":model_rnn,"LSTM":model_lstm,"GRU":model_gru}[name].parameters())
    print(f"  {name:8s}  {best:>8.4f}  {final_ppl:>8.2f}  {np_:>12,}")

print("\nClassification report — LSTM :")
print(classification_report(labels_all, preds_lstm, target_names=CLASSES))


## 6. Système Seq2Seq — Encodeur / Décodeur + Teacher Forcing

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 7. SEQ2SEQ — GÉNÉRATION DE RÉSUMÉS DE SENTIMENT
# ─────────────────────────────────────────────────────────────────────────────
# Tâche : Seq2Seq entraîné à générer un mot-clé résumant le sentiment
# (ex. "positif" → "excellent" / "mitigé" → "moyen" / "négatif" → "mauvais")

# Mini vocabulaire de sortie pour le décodeur
target_vocab = {
    "<PAD>": 0, "<SOS>": 1, "<EOS>": 2,
    "excellent": 3, "bien": 4, "moyen": 5, "mauvais": 6, "décevant": 7
}
SENTIMENT_TO_TARGET = {
    "positif": ["excellent", "bien"],
    "mitigé":  ["moyen"],
    "négatif": ["mauvais", "décevant"]
}
idx2target = {v: k for k, v in target_vocab.items()}
TGT_VOCAB  = len(target_vocab)
TGT_LEN    = 4   # <SOS> + mot + <EOS> + <PAD>

def make_target_seq(sentiment_label):
    """Génère la séquence cible pour un label de sentiment."""
    words = SENTIMENT_TO_TARGET[CLASSES[sentiment_label]]
    ids = [target_vocab["<SOS>"]] + [target_vocab[w] for w in words] + [target_vocab["<EOS>"]]
    ids += [target_vocab["<PAD>"]] * (TGT_LEN - len(ids))
    return ids[:TGT_LEN]

# Dataset Seq2Seq
X_s2s = torch.tensor(X_train[:200], dtype=torch.long)
Y_s2s = torch.tensor([make_target_seq(y) for y in y_train[:200]], dtype=torch.long)
loader_s2s = DataLoader(TensorDataset(X_s2s, Y_s2s), batch_size=16, shuffle=True)

print(f"Exemples de séquences cibles :")
for i in range(3):
    src = CLASSES[y_train[i]]
    tgt = [idx2target[j] for j in make_target_seq(y_train[i])]
    print(f"  Sentiment={src:10s}  →  {tgt}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. ARCHITECTURES ENCODEUR ET DÉCODEUR
# ─────────────────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    """
    Encodeur GRU : lit la séquence source et produit un vecteur de contexte.
    c = dernier état caché h_T de l'encodeur.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.gru   = nn.GRU(embed_dim, hidden_dim, batch_first=True)

    def forward(self, src):
        emb = self.embed(src)              # (B, L, E)
        out, h = self.gru(emb)             # h : (1, B, H)
        return h                           # vecteur de contexte

class Decoder(nn.Module):
    """
    Décodeur GRU : génère la séquence cible token par token.
    Conditionné par le vecteur de contexte de l'encodeur.
    Supporte le Teacher Forcing pendant l'entraînement.
    """
    def __init__(self, tgt_vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed   = nn.Embedding(tgt_vocab_size, embed_dim, padding_idx=0)
        self.gru     = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc      = nn.Linear(hidden_dim, tgt_vocab_size)

    def forward_step(self, token, h):
        """Décode un seul token."""
        emb  = self.embed(token.unsqueeze(1))    # (B, 1, E)
        out, h = self.gru(emb, h)                # (B, 1, H)
        logit = self.fc(out.squeeze(1))          # (B, V)
        return logit, h

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_len=TGT_LEN):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_len = tgt_len

    def forward(self, src, tgt=None, teacher_forcing_ratio=0.5):
        """
        src : (B, L_src)
        tgt : (B, L_tgt) — séquences cibles pour teacher forcing
        teacher_forcing_ratio : probabilité d'utiliser le vrai token
        """
        h = self.encoder(src)                          # (1, B, H)
        B = src.size(0)
        token = torch.full((B,), target_vocab["<SOS>"],
                           dtype=torch.long, device=src.device)
        logits_all = []

        for t in range(1, self.tgt_len):
            logit, h = self.decoder.forward_step(token, h)   # (B, V)
            logits_all.append(logit.unsqueeze(1))
            # Teacher Forcing
            if tgt is not None and torch.rand(1).item() < teacher_forcing_ratio:
                token = tgt[:, t]                   # vrai token
            else:
                token = logit.argmax(1)             # token prédit

        return torch.cat(logits_all, dim=1)         # (B, L-1, V)

ENC = Encoder(VOCAB_REAL, EMBED_DIM, HIDDEN_DIM).to(device)
DEC = Decoder(TGT_VOCAB,  EMBED_DIM, HIDDEN_DIM).to(device)
s2s = Seq2Seq(ENC, DEC).to(device)

n_s2s = sum(p.numel() for p in s2s.parameters())
print(f"Seq2Seq — Paramètres totaux : {n_s2s:,}")
print(f"  Encodeur : {sum(p.numel() for p in ENC.parameters()):,}")
print(f"  Décodeur : {sum(p.numel() for p in DEC.parameters()):,}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. ENTRAÎNEMENT SEQ2SEQ + TEACHER FORCING
# ─────────────────────────────────────────────────────────────────────────────
opt_s2s = optim.Adam(s2s.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)
s2s_losses = []

print("Entraînement Seq2Seq (50 epochs)...")
for ep in range(50):
    s2s.train()
    tl = 0
    for xb, yb in loader_s2s:
        xb, yb = xb.to(device), yb.to(device)
        opt_s2s.zero_grad()
        logits = s2s(xb, tgt=yb, teacher_forcing_ratio=0.5)  # (B, L-1, V)
        # logits: (B, L-1, V)  |  yb[:, 1:]: (B, L-1) (on ignore <SOS>)
        loss = loss_fn(logits.reshape(-1, TGT_VOCAB), yb[:, 1:].reshape(-1))
        loss.backward()
        clip_grad_norm_(s2s.parameters(), max_norm=CLIP_GRAD)
        opt_s2s.step()
        tl += loss.item()
    s2s_losses.append(tl / len(loader_s2s))
    if (ep + 1) % 10 == 0:
        print(f"  Epoch {ep+1:2d}/50  loss={s2s_losses[-1]:.4f}")

torch.save(s2s.state_dict(), "outputs/seq2seq_partie3.pt")
print("Seq2Seq sauvegardé → outputs/seq2seq_partie3.pt")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. STRATÉGIES DE DÉCODAGE : Glouton vs Beam Search
# ─────────────────────────────────────────────────────────────────────────────
def greedy_decode(s2s, src_seq):
    """Décodage glouton : à chaque pas, choisir le token de plus haute probabilité."""
    s2s.eval()
    with torch.no_grad():
        src = torch.tensor(src_seq, dtype=torch.long).unsqueeze(0).to(device)
        h   = s2s.encoder(src)
        token = torch.tensor([target_vocab["<SOS>"]], device=device)
        result = []
        for _ in range(TGT_LEN - 1):
            logit, h = s2s.decoder.forward_step(token, h)
            token = logit.argmax(1)
            word  = idx2target[token.item()]
            if word == "<EOS>": break
            if word not in ("<PAD>", "<SOS>"):
                result.append(word)
    return result

def beam_search_decode(s2s, src_seq, beam_width=3):
    """
    Beam Search : maintenir les `beam_width` meilleures hypothèses.
    
    À chaque pas, étend chaque hypothèse par tous les tokens possibles
    et conserve les `beam_width` hypothèses de plus haute probabilité cumulée.
    """
    s2s.eval()
    with torch.no_grad():
        src = torch.tensor(src_seq, dtype=torch.long).unsqueeze(0).to(device)
        h   = s2s.encoder(src)

        # Initialisation : (log_prob, tokens, état caché)
        beams = [(0.0, [target_vocab["<SOS>"]], h)]
        completed = []

        for _ in range(TGT_LEN - 1):
            new_beams = []
            for score, tokens, h_cur in beams:
                token = torch.tensor([tokens[-1]], device=device)
                logit, h_new = s2s.decoder.forward_step(token, h_cur)
                log_probs = F.log_softmax(logit, dim=-1).squeeze()
                # Top-k tokens
                top_k = torch.topk(log_probs, beam_width)
                for lp, idx in zip(top_k.values, top_k.indices):
                    new_beams.append((score + lp.item(),
                                      tokens + [idx.item()],
                                      h_new))
            # Garder les beam_width meilleures
            new_beams.sort(key=lambda x: x[0], reverse=True)
            beams = new_beams[:beam_width]

        # Meilleure hypothèse
        best_seq = beams[0][1][1:]  # enlève <SOS>
        result   = []
        for idx in best_seq:
            word = idx2target.get(idx, "<UNK>")
            if word == "<EOS>": break
            if word not in ("<PAD>", "<SOS>"):
                result.append(word)
    return result

# ── Test sur quelques exemples ───────────────────────────────────────────────
print("Décodage glouton vs Beam Search sur 10 exemples :\n")
print(f"  {'Sentiment réel':15s}  {'Glouton':15s}  {'Beam (k=3)':15s}")
print("  " + "-" * 48)
for i in range(10):
    src_seq  = X_test[i].tolist()
    true_sent = CLASSES[y_test[i]]
    greedy   = greedy_decode(s2s, src_seq)
    beam     = beam_search_decode(s2s, src_seq, beam_width=3)
    g_str    = " ".join(greedy)  if greedy else "<EOS>"
    b_str    = " ".join(beam)    if beam   else "<EOS>"
    print(f"  {true_sent:15s}  {g_str:15s}  {b_str:15s}")


## 7. Question de Synthèse — Partie III

**Question :** *Dans quelle mesure les architectures récurrentes permettent-elles de modéliser efficacement une séquence réelle, et comment justifier le passage d'un RNN simple vers un LSTM/GRU puis vers un schéma encodeur-décodeur pour une tâche de génération ou de traduction ?*

---

**Modélisation des séquences par les architectures récurrentes :**

Sur le dataset d'avis Blend Burger (365 textes multilingues FR/EN/AR), les architectures récurrentes montrent des capacités différentes selon leur complexité.

Le **RNN simple** souffre du **gradient vanishing** : lors de la rétropropagation à travers le temps (BPTT), le gradient est multiplié T fois par la matrice $W_{hh}$. Si le rayon spectral de $W_{hh}$ est < 1, le gradient s'annule exponentiellement → le modèle ne peut pas capturer des dépendances longue distance (ex. "le service était rapide **mais** la qualité du burger **décevante**").

Le **LSTM** résout ce problème via son mécanisme de cellule ($c_t$) : le chemin du gradient entre $c_t$ et $c_{t-1}$ passe par une **addition** contrôlée par la porte d'oubli, pas une multiplication matricielle. Cela permet de conserver l'information sur plusieurs dizaines de tokens. Le **GRU**, plus simple (pas de cellule séparée), offre des performances similaires avec moins de paramètres (~15% de moins), ce qui est avantageux sur des datasets de petite taille comme le nôtre.

Le **gradient clipping** s'est révélé essentiel même pour LSTM et GRU : sans lui, des pics de gradient peuvent déstabiliser l'entraînement, particulièrement sur des textes arabes (tokenisation plus bruitée).

Le passage au **schéma Seq2Seq** est justifié pour les tâches de génération : la classification produit un label discret, mais la génération ou la traduction nécessite une séquence de sortie de longueur variable. L'encodeur compresse la séquence source en un vecteur de contexte $c$, et le décodeur génère la séquence cible conditionnée par $c$. Le **Teacher Forcing** stabilise l'entraînement en évitant la propagation des erreurs de prédiction.

**Limites observées :** Les RNN récurrents restent contraints à un vecteur de contexte $c$ de dimension fixe (goulot d'information) → sur des textes longs, le mécanisme d'**attention** (Transformer) serait plus adapté.
